# Week 5 Exercise: Personal Knowledge Worker with RAG

## Goal
Build a conversational AI assistant that is an expert on **your** personal information.

### Steps:
1. Assemble personal files into a knowledge base (`my-knowledge-base/`)
2. Vectorize everything into Chroma (vector datastore)
3. Build a conversational AI with Gradio and ask questions!

In [ ]:
# Step 0: Imports and Setup

import os
import glob
import numpy as np
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OPENROUTER_API_KEY not set - please add it to your .env file")

MODEL = "google/gemini-2.5-flash"
DB_NAME = "my_personal_vectordb"

## Step 1: Assemble your Knowledge Base

The `my-knowledge-base/` folder contains personal markdown files:
- `about_me.md` - Background and bio
- `projects.md` - Portfolio of projects
- `skills.md` - Technical skills inventory
- `work_notes.md` - Meeting notes, decisions, and learnings

**Tip:** Replace these with your own files to make it truly personal!

In [ ]:
# Load all documents from our personal knowledge base
KB_PATH = "my-knowledge-base"

loader = DirectoryLoader(KB_PATH, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = loader.load()

print(f"Loaded {len(documents)} documents from {KB_PATH}")
for doc in documents:
    source = os.path.basename(doc.metadata['source'])
    print(f"  - {source} ({len(doc.page_content)} chars)")

## Step 2: Chunk and Vectorize into Chroma

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks")

# Create embeddings using HuggingFace (free, no API key needed)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in Chroma vector database (clear old data if exists)
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)
print(f"Vectorstore created with {vectorstore._collection.count()} vectors")

In [ ]:
# Quick test: verify retrieval works
test_results = vectorstore.similarity_search("What projects have I built?", k=3)
for i, doc in enumerate(test_results):
    print(f"Result {i+1}: {doc.page_content[:120]}...\n")

## Step 3: Build the Conversational AI with RAG

In [ ]:
# Set up retriever and LLM (via OpenRouter)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(
    temperature=0,
    model=MODEL,
    openai_api_key=os.getenv('OPENROUTER_API_KEY'),
    openai_api_base="https://openrouter.ai/api/v1"
)

SYSTEM_PROMPT = """
You are a helpful personal assistant with deep knowledge about your user.
Use the provided context to answer questions accurately.
If the context doesn't contain the answer, say you don't have that information.
Be concise and friendly.

Context:
{context}
"""

def chat(question, history):
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Build prompt with context
    system_message = SYSTEM_PROMPT.format(context=context)

    # Call LLM via OpenRouter
    response = llm.invoke([
        SystemMessage(content=system_message),
        HumanMessage(content=question)
    ])
    return response.content

In [ ]:
# Quick test before launching UI
print(chat("What are my main technical skills?", []))
print("---")
print(chat("Tell me about the CropWatch project", []))

## Step 4: Launch Gradio Chat Interface

In [ ]:
# Launch the chatbot!
gr.ChatInterface(
    chat,
    title="Personal Knowledge Worker",
    description="Ask me anything about your background, projects, skills, and work notes!",
    examples=[
        "What programming languages do I know?",
        "Describe my CropWatch project",
        "What did my team decide about the architecture migration?",
        "What books should I read?",
        "Where did I go to university?"
    ]
).launch(inbrowser=True)